# Sizing & Drawing of Inverter

**Copyright 2026 Simon Dorrer and Harald Pretl**

SPDX-License-Identifier: Apache-2.0 WITH SHL-2.1

# Imports, Constants and Filepaths

In [16]:
# Imports
import math
from spice_params import read_spice_params, write_spice_params

# Constants
k_b = 1.380649e-23 #J/K
gamma = 2/3
T = 130+273.15 # worst case die temperature in Kelvin

# Filepaths
PASSIVES_SIZING ='filterPassivesSizing.spice'

# Specifications

In [17]:
### SYSTEM LEVEL ###

# Start frequency [Hz]
fstart = 100

# Pass-band frequency (-3dB) [Hz]
fpass = 1e6

# Stop-band frequency (fs-fpass) [Hz]
fstop = 9e6

B = fstop - fstart
# Pass-band gain [dB]
Apass = 20

# Pass-band ripple [dB] (allowed)
Rpass = 0.1

# Stop-band attenuation [dB]
astop =49.9

# Maximum Filter order
maxord = 4

# Input/Output structure type
iostruct = "fully differential"

# Maximum number of amplifiers
maxamps = 2

# Max. SNDR (limited by ADC) [dB]
SNDRmax = 48.6

# Noise gain
beta = 1 / (1+10**(Apass/20))

In [18]:
### CHIP LEVEL ###

# Supply voltage
VDD = 1.5

# Input common-mode voltage
Vcm_in = VDD / 2

# Output common-mode voltage
Vcm_out = VDD / 2

# Maximum output signal [V]
Vfs = 1

# Load resistance [Ω]
Rl = 560e3

# Load capacitance [F]
Cl = 5e-12

# Maximum integrated output-referred noise 100Hz to fstop [V]
onoise = 113e-6

# OTA unity-gain bandwidth [Hz]
#fug = 10e6 # naive approach
fug = 30*1*fpass/beta # approach of (10 to 100 -> 30)*Q(1)*fn(fpass)/beta0(1/11)

# Gain error [%]
epsilong = 1e6 # still to be determined

# Maximum chip area [μm²]
area = 200*200

# Maximum total power [W]
Ptot = 100e-6

# Maximum filter THD @ Vfs [dBc]
THDfilt = -60

# Number of bits of the ADC [bits]
nadc = 8

# Passband frequency [Hz]
fpass = 1e6

# Stopband frequency for use with digital decimator [Hz]
fstop_decimator = 9e6

# Stopband frequency [Hz]
fstop = 5e6

In [19]:
# Print Specifications
print("=" * 60)
print("Specifications:")
print("=" * 60)
print(f"Maximum output signal (Vfs) differential: {Vfs} V")
print(f"Load impedance:                           {Rl*1e-3} kΩ || {Cl*1e12} pF")
print(f"Integrated output-referred noise :       ≤{onoise*1e6} μV")
print(f"OTA unity-gain bandwidth fug:            >{fug*1e-6} MHz")
print(f"Gain error εg:                            {epsilong} %")
print(f"Supply voltage VDD:                       {VDD} V")
print(f"Maximum chip area:                       <{area} μm²")
print(f"Total power:                             <{Ptot*1e6} μW")
print(f"Filter THD @ Vfs:                        ≤{THDfilt} dBc")
print(f"Number of bits:                           {nadc}")
print(f"Passband frequency:                       {fpass*1e-6} MHz")
print(f"Stopband frequency w/ dig. decimator:     {fstop_decimator*1e-6} MHz")
print(f"Stopband frequency w/o dig. decimator:    {fstop*1e-6} MHz")
print("=" * 60)

Specifications:
Maximum output signal (Vfs) differential: 1 V
Load impedance:                           560.0 kΩ || 5.0 pF
Integrated output-referred noise :       ≤113.0 μV
OTA unity-gain bandwidth fug:            >330.0 MHz
Gain error εg:                            1000000.0 %
Supply voltage VDD:                       1.5 V
Maximum chip area:                       <40000 μm²
Total power:                             <100.0 μW
Filter THD @ Vfs:                        ≤-60 dBc
Number of bits:                           8
Passband frequency:                       1.0 MHz
Stopband frequency w/ dig. decimator:     9.0 MHz
Stopband frequency w/o dig. decimator:    5.0 MHz


In [ ]:
# Peak SQNR w/ FS sine [dB]
sqnr_peak = 6.02 * nadc + 1.76
# LSB Volage for n bits[V]
Vlsb = Vfs/2**nadc
# Quantization noise for uniform distribution
Vqn = Vlsb/math.sqrt(12)
# Output Swing in RMS
Vorms = Vfs/(2*math.sqrt(2))
# Peak SNR for given noise and output power
SNRpeak = 10*math.log10(Vorms**2/(Vqn**2+onoise**2))

A0_dB = 20*math.log10(fug/fpass)
A0 = 10**(A0_dB/20)
#A0 = gm*R0
#In^2 = 2*4*k_b*T*gamma*gm*B
#In^2 = 2*4*k_b*T*gamma*B*A0/Ro
#Von^2=Ro^2*In^2=2*4*k_b*T*gamma*B*A0*Ro
#Von = Ro*In
#In^2*R0^2=In^2*A0^2/gm^2=8*k_b*T*gamma*B*gm*A0^2/gm^2
#=8*k_b*T*gamma*B*A^2/gm
# fehler in rechnung fixen, letzte rechnung stimmt aber herleitungen passen so nicht
gm_min = 8*k_b*T*gamma*B*(1/beta)**2 / onoise**2
#gm_min = onoise **2 / (2*4*k_b*gamma*(273.15+130)*(fstop-fstart))
Ro_min = A0/gm_min
Cpole = gm_min/(2*math.pi*fug)

In [21]:
# Print Specifications
print("=" * 60)
print("Noise:")
print("=" * 60)
print(f"SQNR_peak:                  {sqnr_peak:.3f} dB")
print(f"Vlsb:                       {Vlsb*1e3:.3f} mV")
print(f"Quantisation noise Voltage: {Vqn*1e3:.3f} mVrms")
print(f"Output voltage:             {Vorms*1e3:.3f} mVrms")
print(f"SNR_peak:                   {SNRpeak:.3f} dB")
print(f"DC-Gain required:           {A0_dB:.3f} dB")
print(f"Required gm:                {gm_min*1e3:.3f} mS")
print(f"Required Ro:                {Ro_min*1e-3:.3f} kOhm")
print(f"Co for pole at fug:         {Cpole*1e12:.3f} pF")
print("=" * 60)

Noise:
SQNR_peak:                  49.920 dB
Vlsb:                       3.906 mV
Quantisation noise Voltage: 1.128 mVrms
Output voltage:             353.553 mVrms
SNR_peak:                   49.882 dB
DC-Gain required:           50.370 dB
Required gm:                2.532 mS
Required Ro:                130.346 kOhm
Co for pole at fug:         1.221 pF


In [22]:
print(fug*1e-6)

330.0


# Load SG13CMOS5L Data Tables

In [23]:
from pygmid import Lookup as lk
import numpy as np
lv_nmos = lk('data/sg13cmos5l_lv_nmos.mat')
lv_pmos = lk('data/sg13cmos5l_lv_pmos.mat')
# list of parameters: VGS, VDS, VSB, L, W, NFING, ID, VT, GM, GMB, GDS, CGG, CGB, CGD, CGS, CDD, CSS, STH, SFL
# if not specified, minimum L, VDS=max(vgs)/2=0.9 and VSB=0 are used 

# Filter Components Sizing

In [24]:
C_per_area = 1.287e-15 # pF/μm² für cmomf
# C = C_per_area * W*L
R_square = 1.645e3 # kΩ/quare für rhigh
# R = R_square * L/W

In [25]:
passives = read_spice_params("filterPassives.spice")
R1 = passives.get("R1")
R2 = passives.get("R2")
R3 = passives.get("R3")
R4 = passives.get("R4")

C1 = passives.get("C1")
C2 = passives.get("C2")
C3 = passives.get("C3")

W_R = 0.5 # 0.5μm
L_R1_exact = R1 / R_square * W_R
L_R2_exact = R2 / R_square * W_R
L_R3_exact = R3 / R_square * W_R
L_R4_exact = R4 / R_square * W_R

L_R1 = round(L_R1_exact / 0.5)*0.5
L_R2 = round(L_R2_exact / 0.5)*0.5
L_R3 = round(L_R3_exact / 0.5)*0.5
L_R4 = round(L_R4_exact / 0.5)*0.5

d_R1 = (L_R1_exact - L_R1) / W_R * R_square
d_R2 = (L_R2_exact - L_R2) / W_R * R_square
d_R3 = (L_R3_exact - L_R3) / W_R * R_square
d_R4 = (L_R4_exact - L_R4) / W_R * R_square

e_R1 = d_R1 / R1 * 100
e_R2 = d_R2 / R2 * 100
e_R3 = d_R3 / R3 * 100
e_R4 = d_R4 / R4 * 100

C_1_Area = C1 / C_per_area
C_2_Area = C2 / C_per_area
C_3_Area = C3 / C_per_area

L_C1_exact = math.sqrt(C_1_Area)
L_C2_exact = math.sqrt(C_2_Area)
L_C3_exact = math.sqrt(C_3_Area)

L_C1 = round(L_C1_exact/0.5)*0.5
L_C2 = round(L_C2_exact/0.5)*0.5
L_C3 = round(L_C3_exact/0.5)*0.5

d_C1 = C1 - C_per_area * L_C1**2
d_C2 = C2 - C_per_area * L_C2**2
d_C3 = C3 - C_per_area * L_C3**2

e_C1 = d_C1 / C1 * 100
e_C2 = d_C2 / C2 * 100
e_C3 = d_C3 / C3 * 100


In [26]:
# Passives area estimation
A_Res_Filter = W_R * (L_R1 + L_R2 + L_R3 + L_R4)
A_Cap_Filter = L_C1**2 + L_C2**2 + L_C3**2
A_Passives_Filter = A_Res_Filter + A_Cap_Filter
L_Passives_equiv = math.sqrt(A_Passives_Filter)

In [27]:
print("=" * 60)
print("Filter Passives Sizing:")
print("=" * 60)
print(f"W R:  {W_R}  μm")
print(f"L R1: {L_R1_exact:.3f} μm, rounded: {L_R1} μm")
print(f"L R2: {L_R2_exact:.3f} μm, rounded: {L_R2} μm")
print(f"L R3: {L_R3_exact:.3f} μm, rounded: {L_R3} μm")
print(f"L R4: {L_R4_exact:.3f} μm, rounded: {L_R4} μm")
print(f"L C1: {L_C1_exact:.3f} μm, rounded: {L_C1} μm")
print(f"L C2: {L_C2_exact:.3f} μm, rounded: {L_C2} μm")
print(f"L C3: {L_C3_exact:.3f} μm, rounded: {L_C3} μm")
print("=" * 60)
print("Deviation from exact target:")
print("=" * 60)
print(f"ΔR1: {d_R1:.3f} Ω, relative: {e_R1:.3f} %")
print(f"ΔR2: {d_R2:.3f} Ω, relative: {e_R2:.3f} %")
print(f"ΔR3: {d_R3:.3f} Ω, relative: {e_R3:.3f} %")
print(f"ΔR4: {d_R4:.3f} Ω, relative: {e_R4:.3f} %")
print(f"ΔC1: {d_C1*1e15:.3f} fF, relative: {e_C1:.3f} %")
print(f"ΔC2: {d_C2*1e15:.3f} fF, relative: {e_C2:.3f} %")
print(f"ΔC3: {d_C3*1e15:.3f} fF, relative: {e_C3:.3f} %")
print("=" * 60)
print("Minimum Passives Area (unreachable - 100% fill)")
print("=" * 60)
print(f"Resistor Area: {A_Res_Filter} μm²")
print(f"Capacitor Area: {A_Cap_Filter} μm²")
print(f"Filter passives Area: {A_Passives_Filter} μm²")
print(f"Filter passives side-lengt (assumes a square): {L_Passives_equiv:.3f} μm")
print("=" * 60)

Filter Passives Sizing:
W R:  0.5  μm
L R1: 3.162 μm, rounded: 3.0 μm
L R2: 31.618 μm, rounded: 31.5 μm
L R3: 5.921 μm, rounded: 6.0 μm
L R4: 48.375 μm, rounded: 48.5 μm
L C1: 139.374 μm, rounded: 139.5 μm
L C2: 19.710 μm, rounded: 19.5 μm
L C3: 27.875 μm, rounded: 28.0 μm
Deviation from exact target:
ΔR1: 532.200 Ω, relative: 5.116 %
ΔR2: 387.100 Ω, relative: 0.372 %
ΔR3: -259.300 Ω, relative: -1.331 %
ΔR4: -410.100 Ω, relative: -0.258 %
ΔC1: -45.342 fF, relative: -0.181 %
ΔC2: 10.618 fF, relative: 2.124 %
ΔC3: -9.008 fF, relative: -0.901 %
Minimum Passives Area (unreachable - 100% fill)
Resistor Area: 44.5 μm²
Capacitor Area: 20624.5 μm²
Filter passives Area: 20669.0 μm²
Filter passives side-lengt (assumes a square): 143.767 μm


## Write sizing to spice param file

In [28]:
passives_sizing = {'W_R': W_R, 'L_R1': L_R1, 'L_R2': L_R2, 'L_R3': L_R3, 'L_R4': L_R4, 'L_C1': L_C1, 'L_C2': L_C2, 'L_C3': L_C3}
write_spice_params(passives_sizing, PASSIVES_SIZING, microns=True)

Wrote 8 parameters to filterPassivesSizing.spice


# OTA Sizing

In [53]:
# Measured DC OP Values for more accurate lookups
viref1 = 0
vo1_n = 0
vo1_p = 0


# Switch for using the educated guess values or using the op data values
sw_use_op = False
if sw_use_op:
    v_ds_dp = (2 * viref1 - vo1_n - vo1_p ) / 2
    vds_dp_load = (vo1_n + vo1_p) / 2
    vds_tail = VDD - viref1
else:
    vds_dp = VDD/2
    vds_dp_load = VDD/2
    vds_tail = VDD/2

### Differential Pair

In [ ]:
gm_id_DP = 15
# Enable Buffer and Enable Switches
# Minimum Size as this is digital
W_EN = 0.15
L_EN = 0.13

L_DP = 0.13                # educated guess, some tradeoff between size / gain / speed
gm_DP = gm_min
id_DP = gm_DP / gm_id_DP
W_DP_exact = id_DP / lv_pmos.lookup('ID_W', GM_ID=gm_id_DP, L=L_DP, VDS=vds_dp, VSB=0)
W_DP = round(W_DP_exact / 0.5) * 0.5
print("-"*60)
print("Differential Pair")
print("-"*60)
print(f"gm/Id:  {gm_id_DP} 1/V")
print(f"gm:     {gm_DP*1e3:.3f} mS")
print(f"Id:     {id_DP*1e6:.3f} μA")
print(f"Length: {L_DP} μm, chosen")
print(f"Width: {W_DP} μm, rounded from: {W_DP_exact:.3f} μm")

L_DP_Load = 0.13
id_DP_Load = id_DP
gm_id_DP_Load = 8
W_DP_Load_exact = id_DP_Load / lv_nmos.lookup('ID_W', GM_ID=gm_id_DP_Load, L=L_DP_Load, VDS=vds_dp, VSB=0)
W_DP_Load = round(W_DP_Load_exact / 0.5) * 0.5
gm_DP_Load = gm_id_DP_Load * id_DP_Load
print("-"*60)
print("Differential Pair Load")
print("-"*60)
print(f"gm/Id:  {gm_id_DP_Load} 1/V")
print(f"gm:     {gm_DP_Load*1e3:.3f} mS")
print(f"Id:     {id_DP_Load*1e6:.3f} μA")
print(f"Length: {L_DP_Load} μm, chosen")
print(f"Width: {W_DP_Load} μm, rounded from: {W_DP_Load_exact:.3f} μm")



SR_1 = 4*math.pi*fug / gm_id_DP
C_c1 = 2*id_DP / SR_1
print(f"Slew-Rate: {SR*1e-6:.3f}V/μs")
print(f"C_C1: {C_c1*1e12:.3f} pF")

------------------------------------------------------------
Differential Pair
------------------------------------------------------------
gm/Id:  15 1/V
gm:     2.532 mS
Id:     219.415 μA
Length: 0.13 μm, chosen
Width: 61.5 μm, rounded from: 61.386 μm
------------------------------------------------------------
Differential Pair Load
------------------------------------------------------------
gm/Id:  8 1/V
gm:     1.755 mS
Id:     219.415 μA
Length: 0.13 μm, chosen
Width: 5.0 μm, rounded from: 5.068 μm
Slew-Rate: 276.460V/μs
C_C1: 1.587 pF


In [63]:
id_Tail = 2 * id_DP

gm_id_Ref = 13
L_Ref = 0.13
id_Ref = i_Tail / 20
gm_Ref = gm_id_Ref * Id_Ref
Vgs_Ref = lv_pmos.look_upVGS(GM_ID=gm_id_Ref, L=L_Ref, VDS=lv_pmos.look_upVGS(GM_ID=gm_id_Ref, L=L_Ref, VDS=0.75, VSB=0), VSB=0)
W_Ref = id_Ref / lv_pmos.lookup("ID_W", GM_ID=gm_id_Ref, L=L_Ref, VDS=Vgs_Ref, VSB=0)

L_Tail = 0.13
gm_id_Tail = lv_pmos.lookup("GM_ID", L=L_Tail, VGS=Vgs_Ref, VDS=vds_tail, VSB=0)
gm_Tail = gm_id_Tail * id_Tail
W_Tail = id_Tail / lv_pmos.lookup("ID_W", GM_ID=gm_id_Tail, L=L_Tail, VDS=vds_tail, VSB=0)

print("Reference Mos-Diode")
print("-"*60)
print(f"gm/Id Ref: {gm_id_Ref:.3f} 1/V")
print(f"gm Ref: {gm_Ref*1e3:.3f} mS")
print(f"Id Ref: {id_Ref*1e6:.3f} μA")
print(f"Vgs Ref: {vgs*1e3:.3f} mV")
print(f"L Ref: {L_Ref:.3f} μm")
print(f"W Ref: {W_Ref:.3f} μm")
print("-"*60)
print("Tail Mos-Mirror")
print("-"*60)
print(f"gm/Id Tail: {gm_id_Tail:.3f} 1/V")
print(f"gm Tail: {gm_Tail*1e3:.3f} mS")
print(f"L Tail: {L_Tail:.3f} μm")
print(f"W Tail: {W_Tail:.3f} μm")
print("-"*60)

Reference Mos-Diode
------------------------------------------------------------
gm/Id Ref: 13.000 1/V
gm Ref: 0.285 mS
Id Ref: 21.942 μA
Vgs Ref: 516.974 mV
L Ref: 0.130 μm
W Ref: 4.335 μm
------------------------------------------------------------
Tail Mos-Mirror
------------------------------------------------------------
gm/Id Tail: 12.517 1/V
gm Tail: 5.493 mS
L Tail: 0.130 μm
W Tail: 73.899 μm
------------------------------------------------------------


In [ ]:
gm_id = 15
# Enable Buffer and Enable Switches
# Minimum Size as this is digital
W_EN = 0.15
L_EN = 0.13

L_DP = 5                # educated guess, some tradeoff between size / gain / speed; still to see is transistor even realizable (too small/big)
gm_DP = gm_min
id_DP = gm_DP / gm_id

print("-"*60)
print("Differential Pair")
print("-"*60)
print(f"gm/Id:  {gm_id} 1/V")
print(f"gm:     {gm_min*1e3} mS")
print(f"Id:     {id_DP*1e6} μA")
# Vgs of NMOS
print(f"Bias Voltages:")
print("-" * 60)
Vgs_NMOS = Vcm_in
print(f"Vgs_NMOS = {Vgs_NMOS} V")

# Vds of NMOS
Vds_NMOS = Vcm_out
print(f"Vds_NMOS = {Vds_NMOS} V")
print("-" * 60 + "\n")

# Transconductance (gm)
print(f"Transconductance (gm):")
print("-" * 60)
gm_id_NMOS = lv_nmos.lookup("GM_ID", L=l, VGS=Vgs_NMOS, VDS=Vds_NMOS, VSB=0)
print(f"gm/ID(L, VGS, VDS, VSB) = {round(float(gm_id_NMOS), 2)} uS/uA")
gm_NMOS = i_out * gm_id_NMOS
print(f"gm_NMOS = {round(float(gm_NMOS) * 1e3, 2)}mS")
print("-" * 60 + "\n")

# Output Conductance (gds)
print(f"Output Conductance (gds):")
print("-" * 60)
gm_gds_NMOS = lv_nmos.lookup("GM_GDS", L=l, VGS=Vgs_NMOS, VDS=Vds_NMOS, VSB=0)
print(f"gm/gds(L, VGS, VDS, VSB) = {round(float(gm_gds_NMOS), 2)}")
gds_NMOS = gm_NMOS / gm_gds_NMOS
print(f"gds_NMOS = {round(float(gds_NMOS) * 1e3, 2)}mS")
print("-" * 60 + "\n")

# Gate capacitance (Cgg), Cgg = Cgs + Cgb + Cdb
print(f"Gate capacitance (Cgg), Cgg = Cgs + Cgb + Cdb:")
print("-" * 60)
gm_cgg_NMOS = lv_nmos.lookup("GM_CGG", L=l, VGS=Vgs_NMOS, VDS=Vds_NMOS, VSB=0)
Cgg_NMOS = gm_NMOS / gm_cgg_NMOS
print(f"Cgg_NMOS = {round(float(Cgg_NMOS) * 1e15, 2)}fF")
print("-" * 60 + "\n")

# Transit Frequency (fT) @ current gain = 1
print(f"Transit Frequency (fT) @ current gain = 1:")
f_T_NMOS = gm_cgg_NMOS / (2 * np.pi)
print(f"f_T_NMOS = {round(float(f_T_NMOS) * 1e-9, 2)}GHz")
print("-" * 60 + "\n")

# Width (W)
print(f"Width (W):")
print("-" * 60)
id_w_NMOS = lv_nmos.lookup("ID_W", L=l, VGS=Vgs_NMOS, VDS=Vds_NMOS, VSB=0)
print(f"ID/W(L, VGS, VDS, VSB) = {round(float(id_w_NMOS * 1e6), 2)} uA/um")
w_NMOS = i_out / id_w_NMOS
print(f"w_NMOS = {round(float(w_NMOS), 2)}um")
print("-" * 60 + "\n")

# Thermal Noise PSD at 1Hz (STH)
print(f"Thermal Noise PSD at 1Hz (STH):")
print("-" * 60)
sth_gm = lv_nmos.lookup('STH_GM', GM_ID=gm_id_NMOS, L=l, VDS=Vds_NMOS, VSB=0)
sth = sth_gm * gm_NMOS
print(f"STH(gm/ID, L, VDS, VSB) = {round(float(sth * 1e24), 2)} pV²/Hz (thermal noise psd at 1 Hz)")
print("-" * 60 + "\n")

# Flicker Corner Frequency (fco)
print(f"Flicker Corner Frequency (fco):")
print("-" * 60)
fco = lv_nmos.lookup('SFL_STH', GM_ID=gm_id_NMOS, L=l, VDS=Vds_NMOS, VSB=0)
print(f"fco(gm/ID, L, VDS, VSB) = {round(float(fco * 1e-6), 2)} MHz (flicker corner frequency @ flicker noise PSD = thermal noise PSD)")
print("=" * 60)

In [ ]:
# PMOS Sizing
print("=" * 60)
print("PMOS Sizing:")
print("=" * 60, "\n")

# Vgs of PMOS
print(f"Bias Voltages:")
print("-" * 60)
Vgs_PMOS = VDD - Vcm_in
print(f"Vgs_PMOS = {Vgs_PMOS} V")

# Vds of PMOS
Vds_PMOS = VDD - Vcm_out
print(f"Vds_PMOS = {Vds_PMOS} V")
print("-" * 60 + "\n")

# Transconductance (gm)
print(f"Transconductance (gm):")
print("-" * 60)
gm_id_PMOS = gm_id_NMOS # lv_pmos.lookup("GM_ID", L=l_MA, VGS=Vgs_PMOS, VDS=Vds_PMOS, VSB=0)
print(f"gm/ID(L, VGS, VDS, VSB) = {round(float(gm_id_PMOS), 2)} uS/uA")
gm_PMOS = i_out * gm_id_PMOS
print(f"gm_PMOS = {round(float(gm_PMOS) * 1e3, 2)}mS")
print("-" * 60 + "\n")

# Output Conductance (gds)
print(f"Output Conductance (gds):")
print("-" * 60)
gm_gds_PMOS = lv_pmos.lookup("GM_GDS", L=l, VGS=Vgs_PMOS, VDS=Vds_PMOS, VSB=0)
print(f"gm/gds(L, VGS, VDS, VSB) = {round(float(gm_gds_PMOS), 2)}")
gds_PMOS = gm_PMOS / gm_gds_PMOS
print(f"gds_PMOS = {round(float(gds_PMOS) * 1e3, 2)}mS")
print("-" * 60 + "\n")

# Gate capacitance (Cgg), Cgg = Cgs + Cgb + Cdb
print(f"Gate capacitance (Cgg), Cgg = Cgs + Cgb + Cdb:")
print("-" * 60)
gm_cgg_PMOS = lv_pmos.lookup("GM_CGG", L=l, VGS=Vgs_PMOS, VDS=Vds_PMOS, VSB=0)
Cgg_PMOS = gm_PMOS / gm_cgg_PMOS
print(f"Cgg_PMOS = {round(float(Cgg_PMOS) * 1e15, 2)}fF")
print("-" * 60 + "\n")

# Transit Frequency (fT) @ current gain = 1
print(f"Transit Frequency (fT) @ current gain = 1:")
f_T_PMOS = gm_cgg_PMOS / (2 * np.pi)
print(f"f_T_PMOS = {round(float(f_T_PMOS) * 1e-9, 2)}GHz")
print("-" * 60 + "\n")

# Width (W)
print(f"Width (W):")
print("-" * 60)
id_w_PMOS = lv_pmos.lookup("ID_W", L=l, VGS=Vgs_PMOS, VDS=Vds_PMOS, VSB=0)
print(f"ID/W(L, VGS, VDS, VSB) = {round(float(id_w_PMOS * 1e6), 2)} uA/um")
w_PMOS = i_out / id_w_PMOS
print(f"w_PMOS = {round(float(w_PMOS), 2)}um")
print("-" * 60 + "\n")

# Thermal Noise PSD at 1Hz (STH)
print(f"Thermal Noise PSD at 1Hz (STH):")
print("-" * 60)
sth_gm = lv_pmos.lookup('STH_GM', GM_ID=gm_id_PMOS, L=l, VDS=Vds_PMOS, VSB=0)
sth = sth_gm * gm_PMOS
print(f"STH(gm/ID, L, VDS, VSB) = {round(float(sth * 1e24), 2)} pV²/Hz (thermal noise psd at 1 Hz)")
print("-" * 60 + "\n")

# Flicker Corner Frequency (fco)
print(f"Flicker Corner Frequency (fco):")
print("-" * 60)
fco = lv_pmos.lookup('SFL_STH', GM_ID=gm_id_PMOS, L=l, VDS=Vds_PMOS, VSB=0)
print(f"fco(gm/ID, L, VDS, VSB) = {round(float(fco * 1e-6), 2)} MHz (flicker corner frequency @ flicker noise PSD = thermal noise PSD)")
print("=" * 60)

In [ ]:
# PMOS / NMOS Width Ratio
w_ratio = w_PMOS / w_NMOS
print(f"PMOS/NMOS Width Ratio (W_PMOS/W_NMOS) = {round(float(w_ratio), 2)}")

# Define Widths and Fingers

In [ ]:
def calculate_finger_options(width1_um, width2_um, target_ratio, min_finger_width=1.0, max_finger_width=20.0, max_options=10, finger_width_step=1.0):
    """
    Calculate practicable finger configurations for two transistors maintaining their width ratio.
    Finger widths are constrained to multiples of finger_width_step (default: 0.5um).
    Only returns configurations with an even number of fingers.
    Both transistors must have the SAME number of fingers.
    
    Parameters:
    -----------
    width1_um : float
        Width of first transistor in micrometers (e.g., PMOS)
    width2_um : float
        Width of second transistor in micrometers (e.g., NMOS)
    target_ratio : float
        Target width ratio (width1 / width2)
    min_finger_width : float
        Minimum practical finger width in micrometers (default: 1.0um)
    max_finger_width : float
        Maximum practical finger width in micrometers (default: 20.0um)
    max_options : int
        Maximum number of options to return (default: 10)
    finger_width_step : float
        Step size for finger width in micrometers (default: 1.0um)
    
    Returns:
    --------
    list of tuples : (nf, fw1, fw2, actual_ratio, error_percent)
        nf: number of fingers (same for both transistors)
        fw1: finger width for transistor 1
        fw2: finger width for transistor 2
        List of viable configurations sorted by error and preference
    """
    
    combinations = []
    
    # Try different even numbers of fingers
    # Start from 2 and go up to a reasonable maximum
    max_nf = int(max(width1_um, width2_um) / min_finger_width) + 2
    
    for nf in range(2, max_nf + 1, 2):  # Only even numbers
        # Calculate required finger widths for this number of fingers
        fw1_exact = width1_um / nf
        fw2_exact = width2_um / nf
        
        # Round to nearest finger_width_step
        fw1 = round(fw1_exact / finger_width_step) * finger_width_step
        fw2 = round(fw2_exact / finger_width_step) * finger_width_step
        
        # Check if both finger widths are within acceptable range
        if (min_finger_width <= fw1 <= max_finger_width and 
            min_finger_width <= fw2 <= max_finger_width):
            
            # Calculate actual widths with rounded finger widths
            actual_width1 = nf * fw1
            actual_width2 = nf * fw2
            
            # Calculate actual ratio
            actual_ratio = actual_width1 / actual_width2
            
            # Calculate percentage error from target ratio
            error_percent = abs(actual_ratio - target_ratio) / target_ratio * 100
            
            # Add to combinations
            combinations.append((nf, fw1, fw2, actual_ratio, error_percent))
    
    # Sort by preference: minimize error (but less aggressive), then prefer good finger widths
    def preference_score(combo):
        nf, fw1, fw2, actual_ratio, error_percent = combo
        # Primary: minimize ratio error (less aggressive weight to allow more error)
        error_score = error_percent * 2  # Reduced from 10 to 2
        # Secondary: prefer finger widths around 3-8um range
        width_score1 = abs(fw1 - 5.0)
        width_score2 = abs(fw2 - 5.0)
        # Tertiary: prefer fewer fingers
        finger_score = nf * 0.01
        return error_score + width_score1 + width_score2 + finger_score
    
    combinations.sort(key=preference_score)
    
    return combinations[:max_options]

In [ ]:
# Calculate finger configurations for all transistors
print("=" * 60)
print('Finger Configuration Options:')
print("=" * 60 + "\n")

# Inverter A
print("Inverter A:")
print("-" * 60)
print(f"Target: PMOS width = {round(float(w_PMOS), 2)}um, NMOS width = {round(float(w_NMOS), 2)}um")
print(f"Target ratio (PMOS/NMOS) = {round(float(w_ratio), 2)}")
print("-" * 60 + "\n")

inverter_a_options = calculate_finger_options(
    round(float(w_PMOS), 2), 
    round(float(w_NMOS), 2), 
    float(w_ratio),
    min_finger_width=1.0,
    max_finger_width=20.0,
    max_options=10,
    finger_width_step=1.0
)

for i, (nf, fw_p, fw_n, actual_ratio, error_pct) in enumerate(inverter_a_options[:5], 1):
    actual_w_p = nf * fw_p
    actual_w_n = nf * fw_n
    print(f"Option {i}:")
    print(f"  Number of fingers: {nf} (same for both)")
    print(f"  PMOS: {nf} fingers × {round(fw_p, 2)}um/finger = {round(actual_w_p, 2)}um")
    print(f"  NMOS: {nf} fingers × {round(fw_n, 2)}um/finger = {round(actual_w_n, 2)}um")
    print(f"  Actual ratio: {round(actual_ratio, 2)}, Error: {round(error_pct, 2)}%")
    print("-" * 60 + "\n")

print("Final Widths for Inverter A:")
# Extract values from the best option (first in the list)
if inverter_a_options:
    NF_PMOS, w_NF_PMOS, w_NF_NMOS, _, _ = inverter_a_options[3]
    w_PMOS = w_NF_PMOS * NF_PMOS
    print(f"w_PMOS = w_NF_PMOS * NF_PMOS = {round(w_NF_PMOS, 2)}um * {NF_PMOS} = {round(w_PMOS, 2)}um")

    NF_NMOS = NF_PMOS  # Same number of fingers for both
    w_NMOS = w_NF_NMOS * NF_NMOS
    print(f"w_NMOS = w_NF_NMOS * NF_NMOS = {round(w_NF_NMOS, 2)}um * {NF_NMOS} = {round(w_NMOS, 2)}um")
else:
    print("No suitable finger configuration found!")
print("=" * 60)

In [ ]:
# NMOS Sizing with final width
print("=" * 60)
print("NMOS Sizing with final width:")
print("=" * 60, "\n")

# Output Current (i_out)
print(f"Output Current (i_out):")
print("-" * 60)
i_out = w_NMOS * id_w_NMOS
print(f"Output current with final width: i_out = w_NMOS * ID/W = {round(float(i_out) * 1e3, 2)} mA")
print("-" * 60 + "\n")

# Transconductance (gm)
print(f"Transconductance (gm):")
print("-" * 60)
gm_NMOS = i_out * gm_id_NMOS
print(f"gm_NMOS = {round(float(gm_NMOS) * 1e3, 2)}mS")
print("-" * 60 + "\n")

# Output Conductance (gds)
print(f"Output Conductance (gds):")
print("-" * 60)
gds_NMOS = gm_NMOS / gm_gds_NMOS
print(f"gds_NMOS = {round(float(gds_NMOS) * 1e3, 2)}mS")
print("-" * 60)

In [ ]:
# PMOS Sizing with final width
print("=" * 60)
print("PMOS Sizing with final width:")
print("=" * 60, "\n")

# Output Current (i_out)
print(f"Output Current (i_out):")
print("-" * 60)
i_out = w_PMOS * id_w_PMOS
print(f"Output current with final width: i_out = w_PMOS * ID/W = {round(float(i_out) * 1e3, 2)} mA")
print("-" * 60 + "\n")

# Transconductance (gm)
print(f"Transconductance (gm):")
print("-" * 60)
gm_PMOS = i_out * gm_id_PMOS
print(f"gm_PMOS = {round(float(gm_PMOS) * 1e3, 2)}mS")
print("-" * 60 + "\n")

# Output Conductance (gds)
print(f"Output Conductance (gds):")
print("-" * 60)
gds_PMOS = gm_PMOS / gm_gds_PMOS
print(f"gds_PMOS = {round(float(gds_PMOS) * 1e3, 2)}mS")
print("-" * 60)

In [ ]:
# Transconductance of inverter
print(f"Transconductance of inverter:")
print("-" * 60)
gm_A = gm_NMOS + gm_PMOS
print(f"gm_A = gm_NMOS + gm_PMOS = {round(float(gm_A) * 1e3, 2)}mS")
print("-" * 60 + "\n")

# Output Conductance of inverter
print(f"Output Conductance of inverter:")
print("-" * 60)
gds_A = gds_NMOS + gds_PMOS
print(f"gds_A = gds_NMOS + gds_PMOS = {round(float(gds_A) * 1e3, 2)}mS")
print("-" * 60 + "\n")

# Open-Loop Gain (Aol) of inverter
print(f"Open-Loop Gain (Aol) of inverter:")
print("-" * 60)
Aol_A = - gm_A / gds_A
print(f"Aol_A = {round(float(Aol_A), 2)} = {round(20 * np.log10(float(abs(Aol_A))), 2)} dB")
print("-" * 60 + "\n")

# Output Resistance (Rout) of inverter
print(f"Output Resistance (Rout) of inverter:")
print("-" * 60)
Rout_A = 1 / gds_A if gds_A != 0 else float('inf')
print(f"Rout_A = 1 / gds_A = {round(float(Rout_A), 2)} Ohms")
print("-" * 60 + "\n")

# Open-Loop Cut-Off Frequency (fcu) of inverter
print(f"Open-Loop Cut-Off Frequency (fcu) of inverter:")
print("-" * 60)
fcu_A = 1 / (2 * np.pi * Rout_A * C_load)
print(f"fcu_A = 1 / (2 * pi * Rout_A * C_load) = {round(float(fcu_A) * 1e-6, 2)} MHz")
print("-" * 60 + "\n")

# Unity-Gain Frequency (fT) of inverter
print(f"Unity-Gain Frequency (fT) of inverter:")
print("-" * 60)
fT_A = gm_A / (2 * np.pi * C_load)
print(f"fT_A = gm_A / (2 * pi * C_load) = {round(float(fT_A) * 1e-6, 2)} MHz")
print("=" * 60)

# Area Estimation



In [ ]:
print("=" * 60)
print('Inverter - Area Estimation:')
print("=" * 60)

area_inverter = w_PMOS * l + w_NMOS * l
print(f"area_inverter = {round(float(area_inverter), 2)}um^2")
print("=" * 60)